In [1]:
import os
import cv2
import time
import torch
import numpy as np
from PIL import Image
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


In [2]:

# =============================
# CONFIGURATION
# =============================
INPUT_DIR = "images"
OUTPUT_DIR = "output_sam2"
CHECKPOINT = "checkpoints/sam2_hiera_large.pt"
MODEL_CONFIG = "configs/sam2/sam2_hiera_l.yaml" # Built-in to the library
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:

# =============================
# LOAD MODEL
# =============================
print(f"Loading SAM 2 on {DEVICE}...")
sam2 = build_sam2(MODEL_CONFIG, CHECKPOINT, device=DEVICE, apply_postprocessing=True)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=32,          # High density to catch nested objects (rice vs bowl)
    points_per_batch=64,
    pred_iou_thresh=0.8,         # Reliability filter
    stability_score_thresh=0.92, # Quality filter
    box_nms_thresh=0.75,         # INCREASED: Allows rice to exist "inside" the bowl box
    min_mask_region_area=2000    # Deletes small crumbs, keeps full food items
)


Loading SAM 2 on cuda...


In [4]:

# =============================
# HELPER: Save colored segment
# =============================
def save_colored_segment(image_rgb, mask, output_path):
    """
    Saves a cropped RGBA image of the segment, tightly bounded.
    The background is transparent; the object retains its original color.
    """
    # Build RGBA: use original color where mask is True, transparent elsewhere
    rgba = np.zeros((*image_rgb.shape[:2], 4), dtype=np.uint8)
    rgba[..., :3] = image_rgb
    rgba[..., 3] = (mask * 255).astype(np.uint8)

    # Tight crop around the masked region
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return
    x0, x1 = xs.min(), xs.max() + 1
    y0, y1 = ys.min(), ys.max() + 1
    cropped = rgba[y0:y1, x0:x1]

    Image.fromarray(cropped, mode='RGBA').save(output_path)


def deduplicate_masks(masks, iou_threshold=0.85):
    """Remove masks that heavily overlap with a larger mask already kept."""
    # Sort by area descending so we always keep the larger one first
    masks = sorted(masks, key=lambda m: m['area'], reverse=True)
    kept = []
    for candidate in masks:
        c_mask = candidate['segmentation']
        is_duplicate = False
        for kept_mask_data in kept:
            k_mask = kept_mask_data['segmentation']
            intersection = np.logical_and(c_mask, k_mask).sum()
            union = np.logical_or(c_mask, k_mask).sum()
            iou = intersection / union if union > 0 else 0
            if iou > iou_threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            kept.append(candidate)
    return kept


# =============================
# PROCESSING LOOP
# =============================
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(image_extensions)]

print(f"Found {len(image_files)} images. Starting segmentation...\n")

total_start = time.time()

for filename in image_files:
    img_path = os.path.join(INPUT_DIR, filename)
    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    t0 = time.time()

    # Generate all masks
    masks = mask_generator.generate(image_rgb)
    masks = deduplicate_masks(masks, iou_threshold=0.85)

    # Keep only segments whose area is >= 30% of the largest segment's area
    if masks:
        max_area = max(m['area'] for m in masks)
        masks = [m for m in masks if m['area'] >= 0.30 * max_area]

    elapsed = time.time() - t0

    # Create a subfolder for each image to keep it organized
    img_output_folder = os.path.join(OUTPUT_DIR, os.path.splitext(filename)[0])
    os.makedirs(img_output_folder, exist_ok=True)

    # Save each segment as a color PNG with transparent background
    for i, mask_data in enumerate(masks):
        mask_bool = mask_data['segmentation'].astype(bool)
        segment_filename = f"segment_{i:03d}.png"
        save_colored_segment(
            image_rgb, mask_bool,
            os.path.join(img_output_folder, segment_filename)
        )

    print(f"[{elapsed:.2f}s] {filename} -> {len(masks)} segments saved to '{img_output_folder}/'")

total_elapsed = time.time() - total_start
print(f"\nAll done! Total time: {total_elapsed:.2f}s. Check the '{OUTPUT_DIR}' folder.")

Found 20 images. Starting segmentation...



d:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\food-calorie-estimator\designing_nd_testing\segmentation_module\sam2\sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (d:\academics\buet_cse_undergrad\3-2\CAPSTONE_PROJECT_REPO\food-calorie-estimator\designing_nd_testing\segmentation_module\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(


[3.88s] bhaat_002.jpg -> 1 segments saved to 'output_sam2\bhaat_002/'
[3.46s] bhaat_003.jpg -> 2 segments saved to 'output_sam2\bhaat_003/'
[4.55s] bhaat_004.jpg -> 2 segments saved to 'output_sam2\bhaat_004/'
[3.15s] bhaat_005.jpg -> 3 segments saved to 'output_sam2\bhaat_005/'
[4.10s] boiled_egg_001.jpg -> 1 segments saved to 'output_sam2\boiled_egg_001/'
[4.40s] boiled_egg_002.jpg -> 2 segments saved to 'output_sam2\boiled_egg_002/'
[4.83s] boiled_egg_003.jpg -> 2 segments saved to 'output_sam2\boiled_egg_003/'


KeyboardInterrupt: 